Christopher Perez Lebron: 120405389

Andre Pham - 118830360

Ian Fuller: 117923346

Shasank Patel: 118976501

Abdulrahman Albeladi - 119271452

USE 3 LATE DAYS

# **CMSC426 Project 2: Panorama Stitching**

# Introduction

The aim of this project is to implement an end-to-end pipeline for panorama stitching. We all use the panorama mode on our smart-phones– you’ll implement a pipeline which does the same basic thing!

This document just provides an overview of what you need to do. For a full breakdown of how each step in the pipeline works, see the [course notes](https://cmsc426.github.io/pano/) for this project.

# System Overview

The system diagram summarizes the panorama-stitching pipeline.

![Panorama-stitching pipeline system diagram](system_diagram.png)

In [ ]:
# Download training images from Google Drive
import gdown

gdown.download_folder(
    id="1U0UxiS3N_25aTd7WxDekug14WDpIXcZX", quiet=True, use_cookies=False
)

In [ ]:
# Check whether the training images were successfully imported
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

train_image = mpimg.imread("/content/test_images/TestSet1/1.jpg")
plt.imshow(train_image)
plt.axis("off")
plt.show()

## Problem Statement

### 1. Detect Corners and Adaptive Non-Maximal Suppression (or ANMS) [25 points]

## ANMS output

ANMS output figure omitted from the public version.

#### Step 1: Corner Detection

The first step in stitching a panorama is extracting corners like most computer vision tasks. Here we will use either Harris corners or Shi-Tomasi corners. Use **cv2.cornerHarris** or **cv2.goodFeaturesToTrack** to implement this part.

In [ ]:
### Corner Detection
# 1) Convert image to gray scale image
# 2) Run harris or other corner detection from cv2 (cv2.cornerHarris OR cv2.goodFeaturesToTrack, etc.)
# Show the corner detection results for one image!!!


import cv2


def detect_corner(img):

    # convert to gray scale
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # extract corners w/ qualty
    corners = []
    dst = cv2.cornerHarris(img, 2, 3, 0.04)
    largest = dst.max()
    for i in range(dst.shape[0]):
        for j in range(dst.shape[1]):
            if dst[i, j] > 0.01 * largest:
                corners.append((j, i))

    return np.array(corners), dst

In [ ]:
# Show you result here


img = cv2.imread("/content/test_images/TestSet2/1.jpg")
corners, _ = detect_corner(img)

for x, y in corners:

    cv2.circle(img, (x, y), 3, (0, 0, 255), -1)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

#### Step 2: Adaptive Non-Maximal Suppression (or ANMS)

The objective of this step is to detect corners such that they are equally distributed across the image in order to avoid weird artifacts in warping.

In a real image, a corner is never perfectly sharp, each corner might get a lot of hits out of the **N**
 strong corners - we want to choose only the **$N_{best}$**
 best corners after ANMS. In essence, you will get a lot more corners than you should! ANMS will try to find corners which are true local maxima. The algorithm for implementing ANMS is given below.

In [ ]:
### Adaptive Non-Maximal Suppression (or ANMS)
# Perform ANMS: Adaptive Non-Maximal Suppression
# Show ANMS output as an image


# ASSUMPTIONS:
# cmap is output from cv2.cornerHarris
# ANMS is implemented completely seperate from part 1
def ANMS(cmap, corners, num_best):

    coordinates = corners

    radius = np.full(shape=(coordinates.shape[0], 1), fill_value=999999999999999)

    coordinates = coordinates.T
    Xarray = coordinates[1]
    Yarray = coordinates[0]

    for i in range(len(Xarray)):
        for j in range(len(Xarray)):

            if cmap[Xarray[j], Yarray[j]] > cmap[Xarray[i], Yarray[i]]:

                ED = (Xarray[j] - Xarray[i]) ** 2 + (Yarray[j] - Yarray[i]) ** 2
                if ED < radius[i]:
                    radius[i] = ED

    # get sorted indicies
    sorted_indices = np.argsort(-radius, axis=0).reshape((1, -1))[0]

    out = np.zeros(shape=(num_best, 2), dtype=int)

    # grab NStrongest
    for i in range(0, num_best):
        out[i] = (Xarray[sorted_indices[i]], Yarray[sorted_indices[i]])

    return out

In [ ]:
# Show you result here

# no ANMS
img = cv2.imread("/content/test_images/TestSet2/1.jpg")
corners, _ = detect_corner(img)
for x, y in corners:
    cv2.circle(img, (x, y), 3, (0, 0, 255), -1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


# w/ ANMS
img = cv2.imread("/content/test_images/TestSet2/1.jpg")
corners, cmap = detect_corner(img)
corners = ANMS(cmap, corners, 30)
for x, y in corners:
    cv2.circle(img, (y, x), 3, (0, 0, 255), -1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

### 2. Feature Descriptors: [15pts]

In the previous step, you found the feature points (locations of the Nbest
 best corners after ANMS are called the feature point locations). You need to describe each feature point by a feature vector, this is like encoding the information at each feature point by a vector. One of the easiest feature descriptor is described next.

Take a patch of **size 40×40**
 centered **(this is very important**) around the keypoint/feature point. Now apply gaussian blur (feel free to play around with the parameters, for a start you can use OpenCV’s default parameters in **cv2.GaussianBlur** command. Now, sub-sample the blurred output (this reduces the dimension) to 8×8
. Then reshape to obtain a 64×1
 vector. Standardize the vector to have zero mean and variance of 1. Standardization is used to remove bias and to achieve some amount of illumination invariance.

In [ ]:
# NOTE: this version off centers any corners near the edge of the image
def feature_descript(gray_img, corners):

    res = []
    for x, y in corners:

        start_row = x - 20
        end_row = x + 20
        if start_row < 0:
            start_row += -start_row
            end_row += -start_row
        if end_row > gray_img.shape[0] - 1:
            end_row -= end_row - gray_img.shape[0] - 1
            start_row -= end_row - gray_img.shape[0] - 1

        start_col = y - 20
        end_col = y + 20
        if start_col < 0:
            start_col += -start_col
            end_col += -start_col
        if end_col > gray_img.shape[1] - 1:
            end_col -= end_col - gray_img.shape[1] - 1
            start_col -= end_col - gray_img.shape[1] - 1

        patch = gray_img[start_row:end_row, start_col:end_col]
        blurred_patch = cv2.GaussianBlur(patch, (3, 3), 0)
        sub_sampled = cv2.resize(blurred_patch, (8, 8), interpolation=cv2.INTER_AREA)

        reshaped = sub_sampled.astype(int).reshape((64, -1))
        final = (reshaped - np.mean(reshaped)) / np.std(reshaped)
        res.append((final, (x, y)))

    return res

In [ ]:
# Show you result here

# show corners we are aiming for
img = cv2.imread("/content/test_images/TestSet2/1.jpg")
corners, cmap = detect_corner(img)
corners = ANMS(cmap, corners, 30)
for x, y in corners:
    cv2.circle(img, (y, x), 3, (0, 0, 255), -1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

# show feature descriptors
img = cv2.imread("/content/test_images/TestSet2/1.jpg")
descriptors = feature_descript(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), corners)
for item, _ in descriptors:
    plt.imshow(item.reshape((8, 8)), cmap="gray")
    plt.axis("off")
    plt.show()

### 3. Feature Matching: [15pts]

In the previous step, you encoded each keypoint by **64×1**
 feature vector. Now, you want to match the feature points among the two images you want to stitch together. In computer vision terms, this step is called as finding feature correspondences between the 2 images. Pick a point in image 1, compute sum of square differences between all points in image 2. Take the ratio of best match (lowest distance) to the second best match (second lowest distance) and if this is below some ratio keep the matched pair or reject it. Repeat this for all points in image 1. You will be left with only the confident feature correspondences and these points will be used to estimate the transformation between the 2 images, also called as Homography. Use the function **cv2.drawMatches** to visualize feature correspondences. Below is an image showing matched features.

## Feature Matching

Feature-matching visualization generated in MATLAB.

In [ ]:
### Feature matching

# Features1 is the output from calling feature_descript() on the corners in img1
# so it consits of (featureVector, location)
# where location is a tuple of form (x,y)


# high level overview:
# find differences[i][j]
# for each corner[i] in image1 only match it w/ sortedSSDs[i][j] if there is a significant difference between sortedSSDs[i][j] and sortedSSDs[i][j+1]
# this makes sure we only grab correspondences that have a high degree of certainty
def feature_match(img1, img2, features1, features2):

    # differences[i][j] = difference between features1[i] and features2[j]
    differences = np.zeros((len(features1), len(features2)))

    # compute SSD for all points
    # for each feature in img1
    for i, (featureVector1, _) in enumerate(features1):
        # for each feature in img2
        for j, (featureVector2, _) in enumerate(features2):
            # for the 64 pixels in each feature
            for k in range(64):
                differences[i, j] += (featureVector1[k][0] - featureVector2[k][0]) ** 2

    # list of cv2.DMatch objects
    dmatches = []

    ratio = 0.66
    # find good matches
    for i in range(differences.shape[0]):
        # obtain argsort of differences[i]
        argsort = np.argsort(differences[i])

        # for all other differences conditionally append them for matches for i
        for index in range(len(argsort) - 1):
            if (
                differences[i, argsort[index]] / differences[i, argsort[index + 1]]
                < ratio
            ):
                dmatches.append(
                    cv2.DMatch(
                        _queryIdx=i,
                        _trainIdx=argsort[index],
                        _distance=differences[i, argsort[index]],
                    )
                )

            else:
                break

    return dmatches

In [ ]:
# Show you result here
# Show you result here

# get feature descriptors for corners in image 1
img1 = cv2.imread("/content/test_images/TestSet2/2.jpg")
initial_corners1, cmap1 = detect_corner(img1)
corners1 = ANMS(cmap1, initial_corners1, 100)
features1 = feature_descript(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), corners1)

# get feature descriptors for corners in image 2
img2 = cv2.imread("/content/test_images/TestSet2/3.jpg")
initial_corners2, cmap2 = detect_corner(img2)
corners2 = ANMS(cmap2, initial_corners2, 100)
features2 = feature_descript(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), corners2)

# run feature_match
matches = feature_match(-1, -1, features1, features2)
print(f"len(matches) {len(matches)}")

# convert corners into key point objects
kp1 = []
for x, y in corners1:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners2:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# run drawMatches to create the final image
out = cv2.drawMatches(
    cv2.cvtColor(img1, cv2.COLOR_BGR2RGB),
    kp1,
    cv2.cvtColor(img2, cv2.COLOR_BGR2RGB),
    kp2,
    matches,
    None,
)

plt.figure(figsize=(16, 12))
plt.imshow(out)
plt.axis("off")
plt.show()

### 4. RANSAC for outlier rejection and to estimate Robust Homography: [20pts]

## RANSAC visualization

The following figure illustrates RANSAC results.

In [ ]:
# Ransac to filter out the wrong matchings and return homography
def RANSAC(match_kp1, match_kp2, N, t, threshold):
    max_inliers = []
    best_points1 = []
    best_points2 = []
    best_H = None
    k = 0

    while (k < N) and (len(max_inliers) < threshold):
        # print(f"i = {k} //// {len(max_inliers)} out of {threshold} inliers", flush=True)
        points1 = []
        points2 = []
        inliers = []

        # Choose four feature pairs at random
        ind = np.random.choice(len(match_kp1), size=4, replace=False)

        for i in ind:
            points1.append(match_kp1[i].pt)
            points2.append(match_kp2[i].pt)

        # Compute the homography matrix h using the selected points
        h = cv2.getPerspectiveTransform(np.float32(points1), np.float32(points2))

        # Compute inliers
        if h is not None:

            for j in range(len(match_kp1)):
                # Convert into homogeneous coordinates
                point1 = np.append(np.float32(match_kp1[j].pt), 1).reshape(3, 1)
                # Compute Hpi
                projected_point = np.dot(h, point1)
                # Avoiding divide by zero
                if projected_point[2] > 0:
                    # Convert into Cartesian coordinates
                    projected_point = projected_point / projected_point[2]
                    projected_point = projected_point[:2].reshape(-1)
                    # Compute ssd
                    ssd = np.sum((match_kp2[j].pt - projected_point) ** 2)
                    # Add inliers
                    if ssd < t:
                        inliers.append((j, ssd))

        # Check if current inlier count is the largest so far
        if len(inliers) > len(max_inliers):
            max_inliers = inliers

        k = k + 1

    # Compute homography on 4 edges with least SSD (According to TA)
    sorted_inliers = sorted(max_inliers, key=lambda x: x[1])

    for i, sdd in sorted_inliers[:4]:
        best_points1.append(match_kp1[i].pt)
        best_points2.append(match_kp2[i].pt)

    best_H = cv2.getPerspectiveTransform(
        np.float32(best_points1), np.float32(best_points2)
    )

    return best_H, max_inliers

In [ ]:
# get feature descriptors for corners in image 1
img1 = cv2.imread("/content/test_images/TestSet1/2.jpg")
initial_corners1, cmap1 = detect_corner(img1)
corners1 = ANMS(cmap1, initial_corners1, 100)
features1 = feature_descript(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), corners1)

# get feature descriptors for corners in image 2
img2 = cv2.imread("/content/test_images/TestSet1/3.jpg")
initial_corners2, cmap2 = detect_corner(img2)
corners2 = ANMS(cmap2, initial_corners2, 100)
features2 = feature_descript(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), corners2)

# run feature_match
matches = feature_match(-1, -1, features1, features2)

# convert corners into key point objects
kp1 = []
for x, y in corners1:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners2:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# Show you result here

# Draw only inlier matches
inlier_matches = [matches[i] for i, ssd in inliers]
print(f"{len(inlier_matches)} inliers")
# Visualize the inliers
out = cv2.drawMatches(
    cv2.cvtColor(img1, cv2.COLOR_BGR2RGB),
    kp1,
    cv2.cvtColor(img2, cv2.COLOR_BGR2RGB),
    kp2,
    inlier_matches,
    None,
)


plt.figure(figsize=(16, 12))
plt.imshow(out)
plt.axis("off")
plt.show()

### 5. Image Warping (and Blending): [25pts]

Panorama can be produced by overlaying the pairwise aligned images to create the final output image. The output panorama stitched from two images shown in the figure below.

When blending these images, there are inconsistency between pixels from different input images due to different exposure/white balance settings or photometric distortions or vignetting. This can be resolved by [Poisson blending](http://www.irisa.fr/vista/Papers/2003_siggraph_perez.pdf). You can use third party code only for the seamless panorama stitching.

The original cell contains an embedded MATLAB-generated image, `Output1.png`.

In [ ]:
# Warp and Blend two images together based on Homography returned in RANSAC
def warp_and_blend(img1, img2, H):
    inv_h = np.linalg.inv(H)  # Inverse to get hg from img2 to img1

    # Get the shape of the images
    h1, w1 = img1.shape[:2]  # Size of img1
    h2, w2 = img2.shape[:2]  # Size of img2

    # Define corners of img2
    corners_img2 = np.array(
        [[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32
    ).reshape(-1, 1, 2)

    # Warp the corners of img2 using the homography
    warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

    # Define the corners of img1 (which is at (0,0) in the panorama space)
    corners_img1 = np.array(
        [[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32
    ).reshape(-1, 1, 2)

    # Combine all corners (from warped img2 and img1) to calculate the bounding box
    all_corners = np.vstack((warped_corners_img2, corners_img1))

    # Find the bounding box for the panorama
    x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
    x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

    # Calculate the size of the output panorama
    output_width = x_max - x_min
    output_height = y_max - y_min

    # Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
    translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])

    # Warp img2 to the new size and translation
    warped_img = cv2.warpPerspective(
        img2, translation_matrix @ inv_h, (output_width, output_height)
    )

    # Create a mask for img2
    mask = np.zeros(warped_img.shape, dtype=np.uint8)
    mask[warped_img > 0] = 255
    mask = mask[:, :, 0]

    # Enlarge img1 by padding with zeros (black) to match the size of the panorama
    img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
    img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1

    # Create a mask for img1 that identifies non-black pixels (valid region in warped_img2)
    mask_img2 = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
    _, mask_img2 = cv2.threshold(mask_img2, 1, 255, cv2.THRESH_BINARY)

    # Create a mask for img2 that identifies non-black pixels (valid region in img2_padded)
    mask_img1 = cv2.cvtColor(img1_padded, cv2.COLOR_BGR2GRAY)
    _, mask_img1 = cv2.threshold(mask_img1, 1, 255, cv2.THRESH_BINARY)

    # Combine the two masks to find the overlapping region (where both masks are 255)
    overlap_mask = cv2.bitwise_and(mask_img1, mask_img2)
    plt.imshow(overlap_mask)
    plt.show()

    # Find the non-overlapping region of img2 using a mask
    non_overlap_mask = cv2.bitwise_xor(mask_img2, overlap_mask)
    plt.imshow(non_overlap_mask)
    plt.show()

    # Add the non-overlapping region of img2 to img1_padded
    non_overlap_region = cv2.bitwise_and(warped_img, warped_img, mask=non_overlap_mask)
    img1_padded = cv2.add(img1_padded, non_overlap_region)
    plt.imshow(img1_padded)
    plt.show()

    # Find the center point for seamlessClone (center of the warped image)
    center_x = int(
        (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
        - x_min
    )
    center_y = int(
        (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
        - y_min
    )
    center = (center_x, center_y)
    print(center)

    # Perform seamless cloning using NORMAL_CLONE
    blend_img = cv2.seamlessClone(
        warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE
    )
    # blend_img = img1_padded

    return blend_img

In [ ]:
# NEW WARPP BLENDDDD
def warp_and_blend(img1, img2, inv_h):

    # Get the shape of the images
    h1, w1 = img1.shape[:2]  # Size of img1
    h2, w2 = img2.shape[:2]  # Size of img2

    # Define corners of img2
    corners_img2 = np.array(
        [[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32
    ).reshape(-1, 1, 2)

    # Warp the corners of img2 using the homography
    warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

    # Define the corners of img1 (which is at (0,0) in the panorama space)
    corners_img1 = np.array(
        [[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32
    ).reshape(-1, 1, 2)

    # Combine all corners (from warped img2 and img1) to calculate the bounding box
    all_corners = np.vstack((warped_corners_img2, corners_img1))

    # Find the bounding box for the panorama
    x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
    x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

    # Calculate the size of the output panorama
    output_width = x_max - x_min
    output_height = y_max - y_min

    # Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
    translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])

    # Warp img2 to the new size and translation
    warped_img = cv2.warpPerspective(
        img2, translation_matrix @ inv_h, (output_width, output_height)
    )

    # Create a mask for img2
    mask = np.zeros(warped_img.shape, dtype=np.uint8)
    mask[warped_img > 0] = 255
    mask = mask[:, :, 0]

    # Enlarge img1 by padding with zeros (black) to match the size of the panorama
    img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
    img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1

    # Create a mask for img1 that identifies non-black pixels (valid region in warped_img2)
    mask_img2 = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
    _, mask_img2 = cv2.threshold(mask_img2, 1, 255, cv2.THRESH_BINARY)

    # Create a mask for img2 that identifies non-black pixels (valid region in img2_padded)
    mask_img1 = cv2.cvtColor(img1_padded, cv2.COLOR_BGR2GRAY)
    _, mask_img1 = cv2.threshold(mask_img1, 1, 255, cv2.THRESH_BINARY)

    # Combine the two masks to find the overlapping region (where both masks are 255)
    overlap_mask = cv2.bitwise_and(mask_img1, mask_img2)
    plt.imshow(overlap_mask)
    plt.show()

    # Find the non-overlapping region of img2 using a mask
    non_overlap_mask = cv2.bitwise_xor(mask_img2, overlap_mask)
    plt.imshow(non_overlap_mask)
    plt.show()

    # Add the non-overlapping region of img2 to img1_padded
    non_overlap_region = cv2.bitwise_and(warped_img, warped_img, mask=non_overlap_mask)
    img1_padded = cv2.add(img1_padded, non_overlap_region)
    plt.imshow(img1_padded)
    plt.show()

    # Find the center point for seamlessClone (center of the warped image)
    center_x = int(
        (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
        - x_min
    )
    center_y = int(
        (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
        - y_min
    )
    center = (center_x, center_y)
    print(center)

    # Perform seamless cloning using NORMAL_CLONE
    blend_img = cv2.seamlessClone(
        warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE
    )
    # blend_img = img1_padded

    return blend_img, translation_matrix

In [ ]:
# get feature descriptors for corners in image 1
img1 = cv2.imread("/content/test_images/TestSet1/1.jpg")
initial_corners1, cmap1 = detect_corner(img1)
corners1 = ANMS(cmap1, initial_corners1, 100)
features1 = feature_descript(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), corners1)

# get feature descriptors for corners in image 2
img2 = cv2.imread("/content/test_images/TestSet1/2.jpg")
initial_corners2, cmap2 = detect_corner(img2)
corners2 = ANMS(cmap2, initial_corners2, 100)
features2 = feature_descript(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), corners2)

# run feature_match
matches = feature_match(-1, -1, features1, features2)

# convert corners into key point objects
kp1 = []
for x, y in corners1:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners2:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)
print(best_h)
# Show you result here
result = warp_and_blend(img1, img2, best_h)
# sharpen_kernel = np.array([[-1, -1, -1],
#                            [-1,  9, -1],
#                            [-1, -1, -1]])
# sharpened_image = cv2.filter2D(result, -1, sharpen_kernel)
# sharp_img = cv2.cvtColor(sharpened_image, cv2.COLOR_BGR2RGB)
# plt.imshow(sharp_img)
# plt.axis("off")
# plt.show()
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
plt.imshow(result)
plt.axis("off")
plt.show()
result2 = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
cv2.imwrite("result2.jpg", result2)

In [ ]:
img2 = cv2.imread("/content/result2.jpg")
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
plt.imshow(img2)
plt.axis("off")
plt.show()

In [ ]:
# MULTIPLYING HOMOGRAPHIES TEST

# COMPUTE HOMOGRAPHY OF IMG1 TO IMG2
# get feature descriptors for corners in image 1
img3 = cv2.imread("/content/test_images/TestSet1/1.jpg")
initial_corners3, cmap3 = detect_corner(img3)
corners3 = ANMS(cmap3, initial_corners3, 100)
features3 = feature_descript(cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY), corners3)

# get feature descriptors for corners in image 2
img4 = cv2.imread("/content/test_images/TestSet1/2.jpg")
initial_corners4, cmap4 = detect_corner(img4)
corners4 = ANMS(cmap4, initial_corners4, 100)
features4 = feature_descript(cv2.cvtColor(img4, cv2.COLOR_BGR2GRAY), corners4)

# run feature_match
matches = feature_match(-1, -1, features3, features4)

# convert corners into key point objects
kp1 = []
for x, y in corners3:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners4:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
print(len(match_kp1))
h1to2, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# COMPUTE HOMOGRAPHY OF IMG2 TO IMG3
# get feature descriptors for corners in image 1
img3 = cv2.imread("/content/test_images/TestSet1/2.jpg")
initial_corners3, cmap3 = detect_corner(img3)
corners3 = ANMS(cmap3, initial_corners3, 100)
features3 = feature_descript(cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY), corners3)

# get feature descriptors for corners in image 2
img4 = cv2.imread("/content/test_images/TestSet1/3.jpg")
initial_corners4, cmap4 = detect_corner(img4)
corners4 = ANMS(cmap4, initial_corners4, 100)
features4 = feature_descript(cv2.cvtColor(img4, cv2.COLOR_BGR2GRAY), corners4)

# run feature_match
matches = feature_match(-1, -1, features3, features4)

# convert corners into key point objects
kp1 = []
for x, y in corners3:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners4:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
print(len(match_kp1))
h2to3, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)


# WARP AND BLEND IMG2 INTO IMG1
img1 = cv2.imread("/content/test_images/TestSet1/1.jpg")
img2 = cv2.imread("/content/test_images/TestSet1/2.jpg")

# inv_best_h = np.linalg.inv(best_h) # 2 to 1
# inv_h2to3 = np.linalg.inv(h2to3) # 3 to 2
# inv_h = np.linalg.inv(best_h @ h2to3) # 3 to 1
inv_h = np.linalg.inv(h1to2)  # Inverse to get hg from img2 to img1
print(f"FIRST inv_h: {inv_h}")

# Get the shape of the images
h1, w1 = img1.shape[:2]  # Size of img1
h2, w2 = img2.shape[:2]  # Size of img2

# Define corners of img2
corners_img2 = np.array([[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Warp the corners of img2 using the homography
warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

# Define the corners of img1 (which is at (0,0) in the panorama space)
corners_img1 = np.array([[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Combine all corners (from warped img2 and img1) to calculate the bounding box
all_corners = np.vstack((warped_corners_img2, corners_img1))

# Find the bounding box for the panorama
x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

# Calculate the size of the output panorama
output_width = x_max - x_min
output_height = y_max - y_min

# Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
translation_matrix1 = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])
print(f"FIRST T: {translation_matrix1}")

# Warp img2 to the new size and translation
warped_img = cv2.warpPerspective(
    img2, translation_matrix1 @ inv_h, (output_width, output_height)
)

# Create a mask for img2
mask = np.zeros(warped_img.shape, dtype=np.uint8)
mask[warped_img > 0] = 255
mask = mask[:, :, 0]

# Enlarge img1 by padding with zeros (black) to match the size of the panorama
img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1

# Create a mask for img1 that identifies non-black pixels (valid region in warped_img2)
mask_img2 = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
_, mask_img2 = cv2.threshold(mask_img2, 1, 255, cv2.THRESH_BINARY)

# Create a mask for img2 that identifies non-black pixels (valid region in img2_padded)
mask_img1 = cv2.cvtColor(img1_padded, cv2.COLOR_BGR2GRAY)
_, mask_img1 = cv2.threshold(mask_img1, 1, 255, cv2.THRESH_BINARY)

# Combine the two masks to find the overlapping region (where both masks are 255)
overlap_mask = cv2.bitwise_and(mask_img1, mask_img2)
plt.imshow(overlap_mask)
plt.show()

# Find the non-overlapping region of img2 using a mask
non_overlap_mask = cv2.bitwise_xor(mask_img2, overlap_mask)
plt.imshow(non_overlap_mask)
plt.show()

# Add the non-overlapping region of img2 to img1_padded
non_overlap_region = cv2.bitwise_and(warped_img, warped_img, mask=non_overlap_mask)
img1_padded = cv2.add(img1_padded, non_overlap_region)
plt.imshow(img1_padded)
plt.show()

# Find the center point for seamlessClone (center of the warped image)
center_x = int(
    (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
    - x_min
)
center_y = int(
    (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
    - y_min
)
center = (center_x, center_y)
print(center)

# Perform seamless cloning using NORMAL_CLONE
blend_img = cv2.seamlessClone(warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE)

plt.imshow(blend_img)
plt.axis("off")
plt.show()


# WARP AND BLEND IMG3 INTO IMG1AND2
img1 = blend_img
img2 = img4

inv_best_h = np.linalg.inv(h1to2)  # 2 to 1
inv_h2to3 = np.linalg.inv(h2to3)  # 3 to 2
inv_h = inv_best_h @ inv_h2to3  # 3 to 1
inv_h = translation_matrix1 @ inv_h
print(f"SECOND inv_h: {inv_h}")
# inv_h = np.linalg.inv(H) # Inverse to get hg from img2 to img1

# Get the shape of the images
h1, w1 = img1.shape[:2]  # Size of img1
h2, w2 = img2.shape[:2]  # Size of img2

# Define corners of img2
corners_img2 = np.array([[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Warp the corners of img2 using the homography
warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

# Define the corners of img1 (which is at (0,0) in the panorama space)
corners_img1 = np.array([[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Combine all corners (from warped img2 and img1) to calculate the bounding box
all_corners = np.vstack((warped_corners_img2, corners_img1))

# Find the bounding box for the panorama
x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

# Calculate the size of the output panorama
output_width = x_max - x_min
output_height = y_max - y_min

# Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
translation_matrix2 = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])

# Warp img2 to the new size and translation
warped_img = cv2.warpPerspective(
    img2, translation_matrix2 @ inv_h, (output_width, output_height)
)

# Create a mask for img2
mask = np.zeros(warped_img.shape, dtype=np.uint8)
mask[warped_img > 0] = 255
mask = mask[:, :, 0]

# Enlarge img1 by padding with zeros (black) to match the size of the panorama
img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1

# Create a mask for img1 that identifies non-black pixels (valid region in warped_img2)
mask_img2 = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
_, mask_img2 = cv2.threshold(mask_img2, 1, 255, cv2.THRESH_BINARY)

# Create a mask for img2 that identifies non-black pixels (valid region in img2_padded)
mask_img1 = cv2.cvtColor(img1_padded, cv2.COLOR_BGR2GRAY)
_, mask_img1 = cv2.threshold(mask_img1, 1, 255, cv2.THRESH_BINARY)

# Combine the two masks to find the overlapping region (where both masks are 255)
overlap_mask = cv2.bitwise_and(mask_img1, mask_img2)
plt.imshow(overlap_mask)
plt.show()

# Find the non-overlapping region of img2 using a mask
non_overlap_mask = cv2.bitwise_xor(mask_img2, overlap_mask)
plt.imshow(non_overlap_mask)
plt.show()

# Add the non-overlapping region of img2 to img1_padded
non_overlap_region = cv2.bitwise_and(warped_img, warped_img, mask=non_overlap_mask)
img1_padded = cv2.add(img1_padded, non_overlap_region)
plt.imshow(img1_padded)
plt.show()

# Find the center point for seamlessClone (center of the warped image)
center_x = int(
    (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
    - x_min
)
center_y = int(
    (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
    - y_min
)
center = (center_x, center_y)
print(center)

# Perform seamless cloning using NORMAL_CLONE
blend_img = cv2.seamlessClone(warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE)

plt.imshow(blend_img)
plt.axis("off")
plt.show()

In [ ]:
# get feature descriptors for corners in image 1
img3 = cv2.imread("/content/result2.jpg")
initial_corners3, cmap3 = detect_corner(img3)
corners3 = ANMS(cmap3, initial_corners3, 100)
features3 = feature_descript(cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY), corners3)

# get feature descriptors for corners in image 2
img4 = cv2.imread("/content/test_images/TestSet1/1.jpg")
initial_corners4, cmap4 = detect_corner(img4)
corners4 = ANMS(cmap4, initial_corners4, 100)
features4 = feature_descript(cv2.cvtColor(img4, cv2.COLOR_BGR2GRAY), corners4)

# run feature_match
matches = feature_match(-1, -1, features3, features4)

# convert corners into key point objects
kp1 = []
for x, y in corners3:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners4:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
print(len(match_kp1))
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# Show you result here
result = warp_and_blend(img3, img4, best_h)
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
# get feature descriptors for corners in image 1
img1 = cv2.imread("/content/result2.jpg")
initial_corners1, cmap1 = detect_corner(img1)
corners1 = ANMS(cmap1, initial_corners1, 100)
features1 = feature_descript(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), corners1)

# get feature descriptors for corners in image 2
img2 = cv2.imread("/content/test_images/TestSet2/3.jpg")
initial_corners2, cmap2 = detect_corner(img2)
corners2 = ANMS(cmap2, initial_corners2, 100)
features2 = feature_descript(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), corners2)

# run feature_match
matches = feature_match(-1, -1, features1, features2)

# convert corners into key point objects
kp1 = []
for x, y in corners1:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners2:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# Show you result here

# Draw only inlier matches
inlier_matches = [matches[i] for i, ssd in inliers]

# Visualize the inliers
out = cv2.drawMatches(
    cv2.cvtColor(img1, cv2.COLOR_BGR2RGB),
    kp1,
    cv2.cvtColor(img2, cv2.COLOR_BGR2RGB),
    kp2,
    inlier_matches,
    None,
)


plt.figure(figsize=(16, 12))
plt.imshow(out)
plt.axis("off")
plt.show()

In [ ]:
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# Show you result here
inv_h = np.linalg.inv(best_h)  # Inverse to get hg from img2 to img1

# Get the shape of the images
h1, w1 = img1.shape[:2]  # Size of img1
h2, w2 = img2.shape[:2]  # Size of img2
print(h1, w1)
print(h2, w2)

# Define corners of img2
corners_img2 = np.array([[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32).reshape(
    -1, 1, 2
)
print(corners_img2)

# Warp the corners of img2 using the homography
warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

# Define the corners of img1 (which is at (0,0) in the panorama space)
corners_img1 = np.array([[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Combine all corners (from warped img2 and img1) to calculate the bounding box
all_corners = np.vstack((warped_corners_img2, corners_img1))

# Find the bounding box for the panorama
x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
x_max, y_max = np.int32(all_corners.max(axis=0).ravel())
print(x_min)
print(x_max)
print(y_min)
print(y_max)

# Calculate the size of the output panorama
output_width = x_max - x_min
output_height = y_max - y_min

# Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])

# Warp img2 to the new size and translation
warped_img = cv2.warpPerspective(
    img2, translation_matrix @ inv_h, (output_width, output_height)
)

# Create a mask for img (all white)
mask = np.zeros(warped_img.shape, dtype=np.uint8)
mask[warped_img > 0] = 255
mask = mask[:, :, 0]

# Enlarge img1 by padding with zeros (black) to match the size of the panorama
img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1


# Find the center point for seamlessClone (center of the warped image)
center_x = int(
    (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
    - x_min
)
center_y = int(
    (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
    - y_min
)
center = (center_x, center_y)
print(center)

plt.imshow(img1_padded)
plt.show()
plt.imshow(warped_img)
plt.show()
plt.imshow(mask)
plt.show()
# Perform seamless cloning using NORMAL_CLONE
blend_img = cv2.seamlessClone(warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE)

# plt.imshow(img1)
# plt.show()
# plt.imshow(img2)
# plt.show()
# plt.imshow(warped_img)
# plt.show()
plt.imshow(blend_img)
plt.show()

### 6. Putting Everything Together (Stitching more than 2 images)

In [ ]:
# Take in a list of images and stitch them together!
def pano_imgs(img_list):
    main_img = img_list[0]

    for next_img in img_list[1:]:
        # get feature descriptors for corners in main_image
        initial_corners1, cmap1 = detect_corner(main_img)
        corners1 = ANMS(cmap1, initial_corners1, 100)
        features1 = feature_descript(
            cv2.cvtColor(main_img, cv2.COLOR_BGR2GRAY), corners1
        )

        # get feature descriptors for corners in next img
        initial_corners2, cmap2 = detect_corner(next_img)
        corners2 = ANMS(cmap2, initial_corners2, 100)
        features2 = feature_descript(
            cv2.cvtColor(next_img, cv2.COLOR_BGR2GRAY), corners2
        )

        # run feature_match
        matches = feature_match(-1, -1, features1, features2)

        # convert corners into key point objects
        kp1 = []
        for x, y in corners1:
            kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

        # convert corners into key point objects
        kp2 = []
        for x, y in corners2:
            kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

        # Extract the matched points
        match_kp1 = []
        match_kp2 = []
        for m in matches:
            match_kp1.append(kp1[m.queryIdx])
            match_kp2.append(kp2[m.trainIdx])

        # RANSAC
        N = 1000
        t = 7
        threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
        best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

        # Warp and Blend into one image
        main_img = warp_and_blend(main_img, next_img, best_h)

        # # Turn back into BGR for input
        # main_img = cv2.cvtColor(main_img, cv2.COLOR_RGB2BGR)

    return main_img

In [ ]:
def pano_imgs(img_list):
    # Compute inverse homography for each image
    inv_H_list = []
    current_img = img_list[0]

    for next_img in img_list[1:]:
        # get feature descriptors for corners in current_img
        initial_corners1, cmap1 = detect_corner(current_img)
        corners1 = ANMS(cmap1, initial_corners1, 100)
        features1 = feature_descript(
            cv2.cvtColor(current_img, cv2.COLOR_BGR2GRAY), corners1
        )

        # get feature descriptors for corners in next img
        initial_corners2, cmap2 = detect_corner(next_img)
        corners2 = ANMS(cmap2, initial_corners2, 100)
        features2 = feature_descript(
            cv2.cvtColor(next_img, cv2.COLOR_BGR2GRAY), corners2
        )

        # run feature_match
        matches = feature_match(-1, -1, features1, features2)

        # convert corners into key point objects
        kp1 = []
        for x, y in corners1:
            kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

        # convert corners into key point objects
        kp2 = []
        for x, y in corners2:
            kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

        # Extract the matched points
        match_kp1 = []
        match_kp2 = []
        for m in matches:
            match_kp1.append(kp1[m.queryIdx])
            match_kp2.append(kp2[m.trainIdx])

        # RANSAC
        N = 1000
        t = 7
        threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
        best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

        inv_H_list.append(np.linalg.inv(best_h))
        current_img = next_img

    # Use inv_H_list to warp and blend
    main_img = img_list[0]
    translations = []

    for i in range(1, len(img_list)):
        current_h = np.eye(3)
        current_t = np.eye(3)

        # Multiply all previous homographies
        for j in range(i):
            # print(f"inv_H {j} BEFORE @ {inv_H_list[j]}")
            current_h = current_h @ inv_H_list[j]

        # Multiple all previous translations
        for k in translations:
            print(f"FIRST T: {k}")
            current_t = k @ current_t

        # Multiply translation into homography
        current_h = current_t @ current_h
        print(f"{i} inv_h: {current_h}")
        # print(f"inv_H AFTER @ {current_h}")
        # Warp and blend
        main_img, new_t = warp_and_blend(main_img, img_list[i], current_h)
        translations.append(new_t)
        plt.imshow(main_img)
        plt.show()

    return main_img

In [ ]:
import glob
import re


# Sort images to read them in order
def extract_number(path):
    # Extract the filename part and use regex to find the number in the image name
    match = re.search(r"(\d+)\.jpg", path)
    return int(match.group(1)) if match else 0


def middle_left_right_order(images):
    result = []
    n = len(images)
    middle = n // 2  # Find the index of the middle image

    # Add the middle image first
    result.append(images[middle])

    # Alternate between left and right of the middle
    left = middle - 1
    right = middle + 1

    while left >= 0 or right < n:
        if right < n:
            result.append(images[right])
            right += 1
        if left >= 0:
            result.append(images[left])
            left -= 1

    return result


# Show your result here
img_list = []
folder_path = "/content/test_images/TestSet1/"
image_paths = glob.glob(folder_path + "*.jpg")
image_paths = sorted(image_paths, key=extract_number)
# image_paths = middle_left_right_order(image_paths)

for path in image_paths:
    img_list.append(cv2.imread(path))
print(image_paths)

result = pano_imgs(img_list)
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
img_list = []
folder_path = "/content/test_images/TestSet2/"
image_paths = glob.glob(folder_path + "*.jpg")
image_paths = sorted(image_paths, key=extract_number)
# image_paths = middle_left_right_order(image_paths)

for path in image_paths:
    img_list.append(cv2.imread(path))
print(image_paths)

result = pano_imgs(img_list)
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
img_list = []
folder_path = "/content/test_images/TestSet3/"
image_paths = glob.glob(folder_path + "*.jpg")
image_paths = sorted(image_paths, key=extract_number)
# image_paths = middle_left_right_order(image_paths)

for path in image_paths:
    img_list.append(cv2.imread(path))
print(image_paths)

# max_count = 100
# count = 0
# while count < max_count:
#   try:
#     result = pano_imgs(img_list)
#   except:
#     count += 1
#     continue
result = pano_imgs(img_list)
# print(f"Retries: {count}")
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.imshow(result)
plt.axis("off")
plt.show()

## Report
You will be graded primarily based on your report.
A demonstration of understanding of the concepts involved in the project are required show the output produced by your code.

Include visualizations of the output of each stage in your pipeline (as shown in the system diagram on page 2), and a description of what you did for each step. Assume that we’re familiar with the project, so you don’t need to spend time repeating what’s already in the course notes. Instead, focus on any interesting problems you encountered and/or solutions you implemented.

Be sure to include the output panoramas for **all three image sets (from the trainingsets)**. Because you have limited time in which to access the “test set” images, we won’t expect in-depth analysis of your results for them.

As usual, your report must be full English sentences, **not** commented code. There is a word limit of 1500 words and no minimum length requirement.

 **TEST SET will be available 4 days before deadline: TBD on Piazza**

#Project 2 Report

1. Corner Detection

We applied the Harris corner detection algorithm to find corners within an image. The image was first converted to greyscale before we applied the corner-detecting algorithm using cv2.cornerHarris. This is basically a method used in the computation of the gradient of the pixel intensity to then detect corners.

The most prominent problem encountered in this project involved the over-detection of corners where regions were highly textured or noisy. One Threshold is applied to select only the most relevant corners. It reduces irrelevant points of corners while keeping them distributed all over the image.

In [ ]:
# @title Output:
img = plt.imread("/content/Report Images/Step1 Images.PNG")
plt.imshow(img)

Detected corners are marked in red in the corner-detected image, highlighting important points in the image.

 2. Adaptive Non-Maximal Suppression (ANMS)

We implemented the algorithm as provided by the pseudocode with minor changes. We brought in the cmap from detect_corners as our corner scores, and we took our corners from detect_corners as our initial set of local maxima. We then implement the algorithm as prescribed.


One of the encountered problems is the selection of too many redundant corners, particularly in denser, high-texture areas. This problem was resolved by fine-tuning the radius calculation to be able to balance better the strength and spatial separation of the corners. Modifying the selection criteria to give preference to those corners that were not only strong but also more evenly distributed helped mitigate the issue of selecting superfluous points.

In [ ]:
# @title Output:
img = plt.imread("/content/Report Images/Step2 Images.PNG")
plt.imshow(img)

The resulting image after the application of ANMS contains fewer, yet well-scattered, corners marked in red. These are more likely to yield good matches for stitching.

 3. Feature Descriptors

we implemented the algorithm as described in the lecture slides with minor modifications for edge cases

Corners detected near the edge of the image became a challenge for our feature descriptor function because it could result in incomplete patches. This meant that feature descriptors would not have been computed properly, and this reduced the efficiency of matching. In such cases, we slightly shift the 40x40 window so that the pixel is no longer centered. We found that this performed a bit better than simply padding the edges of the image with zeros and keeping the pixel centered in the initial feature descriptor patch. Dealing with this edge case ensures that we always have a 40x40 patch to work with which prevents errors in the feature descriptor computation.


The following images show examples of the 8×8 patches centered around every corner (except when the corner is less than 20 pixels away from any edge, then the window is shifted as described earlier), highlighting some important areas in the image. Also, every patch is normalized so that they would look the same under different lighting conditions.

In [ ]:
# @title Output:
img = plt.imread("/content/Report Images/feature descript.png")
plt.imshow(img)

4. Feature Matching

We implemented the algorithm as described in the lecture slides.

The main problem that was encountered was that the feature descriptors of the image in many regions were very similar, hence giving lots of incorrect matches. This was overcome by tweaking the ratio test, which filtered out weak matches, retaining only those that showed strong correspondence. Further, only the nearest neighborhood in the image was searched, reducing the false match possibilities due to large spatially separated features having similar descriptors. These modified feature matchings worked much better and resulted in much-improved accuracy and reliability.

In [ ]:
# @title Output:
img = plt.imread("/content/Report Images/Step4 Images.PNG")
plt.imshow(img)

The output displays all 3 training sets with there matched points between the two images.Lines connecting corresponding points are drawn. These matches have been filtered to be the most confident ones.



 5. RANSAC

We implemented the algorithm as specified in the lecture slides.

Of course, the challenge was to come up with the optimal number of iterations or N and threshold in defining inliers. If the number of iterations is too low, then poor homography estimates are produced. If they are too high, the process is unnecessarily slowed down. After several experiments, we adopted the threshold to be int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold, and the N to be 1000. Dynamic adjustment of the inlier threshold, as a percentage of the total matches, yielded good results in improving the accuracy of homography estimation.

In [ ]:
# @title Output:
img = plt.imread("/content/Report Images/Step5 Images.png")
plt.imshow(img)

This is the RANSAC output for all the training set, the image that shows only the inlier matches, which are used in order to compute the homography matrix. This matrix will give the mapping of points from one image to the other so proper alignment can be made.

 6. Image Warping and Blending

This function takes two image parameters and performs a reverse homography to transform img2 into img. Corners are warped on img2 based on the homography. The bounding box is calculated and found from the combination of all corners from img1 and warped img2. This allows a panorama space to be created to warp img2 based on a translation matrix. Then a mask is created for both images to find non-black pixels, overlapping regions, and non-overlapping regions and is properly blended using seamlessClone.

A problem that arose from this function is before, the resulting image was very blacked out and it was likely due img1 being padded with too many zeros (blacks) to account for the panorama which caused blending to blacken the lower half of the image.

In [ ]:
# @title Output:
img = plt.imread("/content/Report Images/ImageWarp.PNG")
plt.imshow(img)

Output image alignment demonstrates the accuracy of homography calculations. No varying brightness in different parts of the image, showing successful use of seamlessClone. The black regions around the image, especially upper-left, are a result of panorama space being larger than combined images.
7. Image Stitching/Panorama

To implement the last function, we put everything together into a for loop. Our initial implementation warped and blended each image pair and then searched for correspondences between the warped image and the next image, however, this caused issues with corner detection so we changed our approach a little bit. We computed homographies and translation matrices between image n and image n+1 and inserted them into a list. At the end once we had all of the homographies and translation matrices, we iteratively stitched images together using the corresponding homography and translation matrices. This led us to the successful completion of the training set 2  but left us with major distortion issues in set 3. We believe that the level of distortion caused errors in our warping function due to parts of the last image going out of the bounds of the canvas. All day on Sunday 10/20/24 We all tried to find the source of the error preventing us from fully stitching set 3 and could not find a solution. Eventually, we had routed out any possible cause we could think of and the only thing that was left to do was to implement cylindrical projection. However, after taking a look at this for a while and planning our initial implementation we noticed that this would be very difficult and could spawn a whole host of new bugs to squash. Since we were on our 3rd late token we all decided to cut our losses and turn the project in as we know we all did everything we could to get the project fully functional.

#Output Panorama Sets

In [ ]:
# @title Output: For Set 1
img = plt.imread("/content/Report Images/TestSet1.PNG")
plt.imshow(img)

In [ ]:
# @title Output: For Set 2
img = plt.imread("/content/Report Images/TestSet2.PNG")
plt.imshow(img)

In [ ]:
# @title Output: For Set 3
img = plt.imread("/content/Report Images/TestSet3.PNG")
plt.imshow(img)

# Allowed Functions

For cv2 advanced functions, only these are allowed:
1. For feature detection: **cv2.cornerHarris**, **cv2.cornerHarris**
2. For drawing matches: **cv2.drawMatches**
3. For estimate homograhy and warping: **cv2.getPerspectiveTransform**, **cv2.warpPerspective**

# Submission Guidelines

**If your submission does not comply with the following guidelines, you’ll be given ZERO credit.**

Your submission on ELMS(Canvas) must be a pdf file, following the naming convention **YourDirectoryID_proj2.pdf**. For example, xyz123_proj2.pdf.

**All your results and report should be included in this notebook. After you finished all, please export the notebook as a pdf file and submit it to ELMS(Canvas).**

# Collaboration Policy
You are encouraged to discuss the ideas with your peers. However, the code should be your own, and should be the result of you exercising your own understanding of it. If you reference anyone else’s code in writing your project, you must properly cite it in your code (in comments) and your writeup. For the full honor code refer to the CMSC426 Fall 2023 website.